# SafeRoute AI — Notebook 04: Spatiotemporal Hotspot Identification & Expanding Lagged Risk Formulation

## 1. Context & Methodological Safeguards
This notebook implements the spatiotemporal clustering and rolling lagged risk dataset for **SafeRoute AI**:
> **"Predict/rank high-risk LOCATION + TIME combinations using strictly lagged pre-event historical information."**

### Core Methodological Principles:
1. **Strictly Lagged Pre-Event Features:** Every predictive feature is calculated using **data occurring strictly before the target period**:
   $$\text{MAX}(\text{feature\_data\_date}) < \text{target\_period\_start\_date}$$
2. **Spatial Zone Integrity:** Hotspot zones are discovered using the historical training period (2022-01 to 2024-06) and then held fixed for out-of-time validation/testing.
3. **Burn-In Partition:** The first 6 months (2022-01 to 2022-06) serve as an initial burn-in pool to establish 180-day baseline lag statistics and are excluded from training rows.
4. **Target Variable Formulation:** The **Proposed Severity-Weighted Risk Index (SWRI)** is evaluated strictly on future actual outcomes in each monthly evaluation slice.
5. **Post-Event Context Quarantined:** Contemporaneous weather/traffic crash ratios are excluded from predictive inputs and kept solely for IBM Bob explanations.


In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.neighbors import NearestNeighbors
import folium
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

DATA_PATH = "../data/processed/cleaned_accidents.csv"
df = pd.read_csv(DATA_PATH)
df['date_dt'] = pd.to_datetime(df['date'])

print(f"Loaded clean dataset: {df.shape}")
print(f"Date Range: {df['date_dt'].min().strftime('%Y-%m-%d')} to {df['date_dt'].max().strftime('%Y-%m-%d')}")


Loaded clean dataset: (20000, 24)
Date Range: 2022-01-01 to 2025-04-15


## 2. Spatial Hotspot Zone Discovery (Training Period Only)
Zones are discovered using the historical training period (2022-01-01 to 2024-06-30) via HDBSCAN (`min_cluster_size=10, min_samples=5`) and held fixed.


In [4]:
# Load generated Zone Metadata
zone_meta_df = pd.read_csv('../data/processed/hotspot_zones_metadata.csv')
print(f"Total Persistent Hotspot Zones (fitted on Train): {len(zone_meta_df)}")
display(zone_meta_df.head(10))


Total Persistent Hotspot Zones (fitted on Train): 391


,zone_id,city,center_lat,center_lon,max_r_km,train_period_accidents
0,0,Bangalore,12.819534,77.634097,5.299603,30
1,1,Bangalore,12.844182,77.652003,1.971717,10
2,2,Bangalore,12.825852,77.542592,4.796075,55
3,3,Bangalore,13.177301,77.522766,3.014076,10
4,4,Bangalore,12.806570,77.437297,2.216714,11
5,5,Bangalore,13.055663,77.473985,2.986972,12
6,6,Bangalore,13.002538,77.454604,2.647491,15
7,7,Bangalore,12.892995,77.785196,3.640230,18
8,8,Bangalore,12.860727,77.745380,6.489507,72
9,9,Bangalore,12.869233,77.684046,3.779516,36


## 3. Rolling & Expanding Lagged Feature Architecture
For every target monthly slice and 4-hour window, features are extracted exclusively from prior historical data:
- **Rolling Windows:** `lag_accidents_30d`, `lag_accidents_90d`, `lag_accidents_180d`
- **Recent Trend:** `lag_trend_90d_vs_180d` $= \text{acc\_90d} - (\text{acc\_180d} \times 0.5)$
- **Cumulative Zone History:** `lag_expanding_accidents`, `lag_expanding_fatal_rate`, `lag_expanding_major_rate`, `lag_expanding_casualty_density`
- **Cumulative Zone-Window History:** `lag_zw_expanding_accidents`, `lag_zw_expanding_swri`


In [5]:
# Load final dataset and inspect schema
dataset_path = "../data/processed/spatiotemporal_risk_dataset.csv"
spatiotemporal_df = pd.read_csv(dataset_path)

print(f"Dataset Shape: {spatiotemporal_df.shape}")
print(f"Split Distribution:")
print(spatiotemporal_df['data_split'].value_counts())

print(f"Target Risk Tier Distribution (Excluding Burn-in):")
usable_df = spatiotemporal_df[spatiotemporal_df['data_split'] != 'BURN_IN']
print(pd.crosstab(usable_df['data_split'], usable_df['target_risk_tier'], normalize='index').round(4) * 100)


Dataset Shape: (93840, 33)
Split Distribution:
data_split
TRAIN      56304
BURN_IN    14076
VAL        14076
TEST        9384
Name: count, dtype: int64
Target Risk Tier Distribution (Excluding Burn-in):
target_risk_tier  HIGH    LOW  MEDIUM
data_split                           
TEST              6.52  87.06    6.42
TRAIN             7.68  84.01    8.32
VAL               7.36  84.85    7.79


## 4. Formal Temporal Leakage Audit
We verify that **for 100% of rows**, the feature cutoff date is strictly before the target start date.


In [6]:
# Leakage Verification Check
leakage_violations = (
    pd.to_datetime(spatiotemporal_df['feature_cutoff_date']) >=
    pd.to_datetime(spatiotemporal_df['target_start_date'])
).sum()

print("="*60)
print("FORMAL TEMPORAL LEAKAGE AUDIT RESULT")
print("="*60)
print(f"Total Rows Evaluated: {len(spatiotemporal_df):,}")
print(f"Leakage Violations [MAX(feature_date) >= target_date]: {leakage_violations}")
print("Status: 100% LEAKAGE-FREE (Strictly Temporal)")

sample_audit = spatiotemporal_df[['target_month', 'target_start_date', 'target_end_date', 'feature_cutoff_date', 'zone_id', 'time_window_label', 'lag_accidents_90d', 'lag_expanding_accidents', 'target_accidents', 'target_risk_tier']].sample(5, random_state=42)
display(sample_audit)


FORMAL TEMPORAL LEAKAGE AUDIT RESULT
Total Rows Evaluated: 93,840
Leakage Violations [MAX(feature_date) >= target_date]: 0
Status: 100% LEAKAGE-FREE (Strictly Temporal)


,target_month,target_start_date,target_end_date,feature_cutoff_date,zone_id,time_window_label,lag_accidents_90d,lag_expanding_accidents,target_accidents,target_risk_tier
56258,2023-12,2023-12-01,2023-12-31,2023-11-30,383,08:00 - 11:59 (Morning Rush),1.0,10.0,0.0,LOW
25803,2022-11,2022-11-01,2022-11-30,2022-10-31,390,12:00 - 15:59 (Afternoon),2.0,9.0,0.0,LOW
72586,2024-07,2024-07-01,2024-07-31,2024-06-30,367,16:00 - 19:59 (Evening Rush),9.0,52.0,0.0,LOW
43580,2023-07,2023-07-01,2023-07-31,2023-06-30,225,08:00 - 11:59 (Morning Rush),1.0,9.0,0.0,LOW
6561,2022-03,2022-03-01,2022-03-31,2022-02-28,311,12:00 - 15:59 (Afternoon),0.0,0.0,0.0,LOW


## 5. Feature Classification & Data Roles

| Feature Group | Column Names | Data Source & Pre-Event Timing |
| :--- | :--- | :--- |
| **A. Identifiers & Dates** | `zone_id`, `city`, `data_split`, `target_month`, `target_start_date`, `target_end_date`, `feature_cutoff_date`, `time_window_4h`, `time_window_label`, `is_peak_window` | Schedule & calendar constants |
| **B. Spatial Centroid** | `center_lat`, `center_lon`, `max_r_km` | Fixed from Training HDBSCAN fit |
| **C. Rolling & Expanding Lags** | `lag_accidents_30d`, `lag_accidents_90d`, `lag_accidents_180d`, `lag_trend_90d_vs_180d`, `lag_expanding_accidents`, `lag_expanding_fatal_rate`, `lag_expanding_major_rate`, `lag_expanding_casualty_density`, `lag_zw_expanding_accidents`, `lag_zw_expanding_swri` | **Strictly historical:** computed ONLY on crashes where $\text{date} < \text{target\_start\_date}$ |
| **D. Target Outcomes** | `target_accidents`, `target_fatal`, `target_major`, `target_minor`, `target_casualties`, `target_swri_score`, `target_risk_tier` | **Future labels:** crashes in $[	ext{start}, 	ext{end}]$ |
| **E. Post-Event Explanatory Data** | `actual_rain_fog_ratio`, `actual_low_visibility_ratio`, `actual_high_traffic_ratio` | Quarantined for IBM Bob explainability, NOT in ML feature inputs |
